# Multinomial Naive Bayes

Multinomial Naive Bayes is a probabilistic classification algorithm based on Bayes' theorem. It's particularly well-suited for text classification tasks where the features represent word counts or term frequencies.

## Key Concepts:

- **Multinomial Distribution**: Assumes features follow a multinomial distribution (counts)
- **Independence Assumption**: Assumes features are independent given the class
- **Text Classification**: Commonly used for document classification, spam detection, etc.

## When to Use:

- Text classification with word counts/tf-idf features
- Discrete count data
- When computational efficiency is important

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Load Dataset

We'll use the 20 Newsgroups dataset, a classic text classification dataset.

In [ ]:
# Load a subset of 20 newsgroups dataset
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

# Fetch training data
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers', 'footers', 'quotes'))

print(f"Training samples: {len(newsgroups_train.data)}")
print(f"Test samples: {len(newsgroups_test.data)}")
print(f"\nCategories: {newsgroups_train.target_names}")
print(f"\nSample document:\n{newsgroups_train.data[0][:500]}...")

## Feature Extraction

Convert text data into numerical features using CountVectorizer (word counts).

In [ ]:
# Create count vectorizer
count_vectorizer = CountVectorizer(max_features=5000, stop_words='english')

# Fit and transform training data
X_train_counts = count_vectorizer.fit_transform(newsgroups_train.data)
X_test_counts = count_vectorizer.transform(newsgroups_test.data)

# Apply TF-IDF transformation (optional but often improves performance)
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)

print(f"Training feature matrix shape: {X_train_tfidf.shape}")
print(f"Test feature matrix shape: {X_test_tfidf.shape}")
print(f"Number of features (words): {X_train_tfidf.shape[1]}")

## Train Multinomial Naive Bayes Model

In [ ]:
# Initialize and train the model
mnb = MultinomialNB(alpha=1.0)  # alpha is Laplace smoothing parameter
mnb.fit(X_train_tfidf, newsgroups_train.target)

# Make predictions
y_pred = mnb.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(newsgroups_test.target, y_pred)
print(f"Accuracy: {accuracy:.4f}")

## Model Evaluation

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(newsgroups_test.target, y_pred, target_names=newsgroups_test.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(newsgroups_test.target, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=newsgroups_test.target_names,
            yticklabels=newsgroups_test.target_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Multinomial Naive Bayes')
plt.tight_layout()
plt.show()

## Feature Importance

Examine which words are most important for each category.

In [ ]:
# Get feature names
feature_names = count_vectorizer.get_feature_names_out()

# Get log probabilities for each class
    

In [ ]:
# Display top 10 important words for each category
n_top = 10

for i, category in enumerate(newsgroups_train.target_names):
    # Get indices of top features for this class
    top_indices = np.argsort(mnb.feature_log_prob_[i])[-n_top:][::-1]
    top_words = [feature_names[idx] for idx in top_indices]
    
    print(f"\n{category}:")
    print(f"Top words: {', '.join(top_words)}")

## Hyperparameter Tuning

Experiment with different alpha values (Laplace smoothing).

In [ ]:
# Test different alpha values
alpha_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
accuracies = []

for alpha in alpha_values:
    mnb = MultinomialNB(alpha=alpha)
    mnb.fit(X_train_tfidf, newsgroups_train.target)
    y_pred = mnb.predict(X_test_tfidf)
    acc = accuracy_score(newsgroups_test.target, y_pred)
    accuracies.append(acc)
    print(f"Alpha: {alpha}, Accuracy: {acc:.4f}")

# Plot accuracy vs alpha
plt.figure(figsize=(10, 6))
plt.plot(alpha_values, accuracies, marker='o', linewidth=2)
plt.xlabel('Alpha (Smoothing Parameter)')
plt.ylabel('Accuracy')
plt.title('Multinomial NB: Accuracy vs Alpha')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Predict on New Documents

In [ ]:
# Function to predict category for new text
def predict_category(text):
    # Transform the text
    text_counts = count_vectorizer.transform([text])
    text_tfidf = tfidf_transformer.transform(text_counts)
    
    # Predict
    prediction = mnb.predict(text_tfidf)[0]
    probabilities = mnb.predict_proba(text_tfidf)[0]
    
    # Get class with highest probability
    category = newsgroups_train.target_names[prediction]
    
    return category, probabilities

# Test with sample texts
test_texts = [
    "The graphics card in my computer is not working properly",
    "Jesus Christ is the central figure of Christianity",
    "The patient was diagnosed with a rare medical condition",
    "I don't believe in any religious doctrine"
]

for text in test_texts:
    category, probs = predict_category(text)
    print(f"\nText: {text}")
    print(f"Predicted: {category}")
    print("Probabilities:")
    for name, prob in zip(newsgroups_train.target_names, probs):
        print(f"  {name}: {prob:.4f}")

## Summary

### Key Takeaways:

1. **Multinomial Naive Bayes** is excellent for text classification with count-based features
2. **Laplace smoothing (alpha)** prevents zero probabilities and handles unseen words
3. **Fast training and prediction** - suitable for large datasets
4. **Works well with TF-IDF** features for better performance

### Advantages:
- Simple and fast
- Works well with high-dimensional data
- Good baseline for text classification
- Handles sparse data efficiently

### Limitations:
- Assumes feature independence (often violated in text)
- Sensitive to feature scaling
- May not perform as well as more complex models on some tasks